# Topical breadth and author topic consistency

Sections 4.1.2 and 4.2.1; Appendix A.2 and Table 3. The default run reuses saved numeric artifacts and recomputes the GMM and paper/author statistics. It never loads a trained model or fits BERTopic. Only aggregate numeric results are printed and exported.

Private inputs, all gitignored:

- `data/{pred,norm}/papers.csv`: dataset-local `id` (unique integer), nullable `title` and `description` (abstract strings).
- `data/{pred,norm}/paper_authors.csv`: integer `paper_id`, `author_id`; duplicate links collapse. Author names are not needed.
- `data/topics/seed_manifest.json`: `seeds`, the ordered list of five distinct integer draws, and `main_seed`.
- Each `data/topics/seed_<seed>/` contains `assignments.csv` (`paper_id`, `dataset`, nullable numeric `topic_id`, `weighted_score`); `topic_scores.csv` (`topic_id`, `hier_score`, `hier_norm`); `topic_embeddings.npz` (aligned numeric `topic_ids`, `embeddings` arrays); and `document_mapping.csv` (`paper_id`, `dataset`, `document_sha256`). Dataset values are Predatory or Normal. SHA-256 hashes validate the exact title+abstract documents and order without displaying text.

Topic scores, embeddings and assignments derive from private text. Keep them in the ignored data directory; none is distributed. No topic words, names or representative documents are needed.

## Configuration and random-seed generation

Replay the recorded five seeds by default, including the main seed. To generate new seeds, set `SEED_MODE = "random"`: `random.SystemRandom().sample` draws five distinct integers and saves them before any fitting. SystemRandom is deliberately nondeterministic; the saved list enables replay. New seeds need either their saved artifacts or an explicitly enabled training run. Missing artifacts never trigger training automatically.

The optional training branch preserves source model parameters and uses only the embedding model name `all-mpnet-base-v2`. The recorded training environment used Python 3.13.15 and CUDA; notebook 01 and this saved-artifact path use Python 3.11.17. The requirements file separates their conflicting numerical-library pins with Python-version markers. The historical manifest is not a full transitive lock or model-revision record; retraining equality is not guaranteed.

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import re
import sys
from itertools import combinations
import numpy as np
import pandas as pd
from scipy.stats import norm, ttest_ind, mannwhitneyu
from sklearn.mixture import GaussianMixture
from sklearn.metrics.pairwise import cosine_similarity

REPO_ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
DATA_DIR = REPO_ROOT / 'data'
TOPIC_DIR = DATA_DIR / 'topics'
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'topical_breadth'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TOPIC_MODEL = False
SEED_MODE = 'replay'
MAIN_SEED = 1644304431
DATASETS = {'Normal':'norm', 'Predatory':'pred'}

if SEED_MODE == 'random':
    SEEDS = random.SystemRandom().sample(range(1, 2**31 - 1), 5)
    MAIN_SEED = SEEDS[0]
    TOPIC_DIR.mkdir(parents=True, exist_ok=True)
    manifest_path = TOPIC_DIR / 'generated_seed_manifest.json'
    if manifest_path.exists():
        raise FileExistsError('Preserve the existing generated seed manifest before drawing another batch.')
    manifest_path.write_text(json.dumps({'seeds':SEEDS, 'main_seed':MAIN_SEED}, indent=2), encoding='utf-8')
elif SEED_MODE == 'replay':
    manifest = json.loads((TOPIC_DIR / 'seed_manifest.json').read_text(encoding='utf-8'))
    SEEDS = manifest['seeds']
    if MAIN_SEED != manifest['main_seed']:
        raise ValueError('Main seed and saved seed manifest disagree.')
else:
    raise ValueError('SEED_MODE must be replay or random.')
assert len(SEEDS) == len(set(SEEDS)) == 5
assert all(type(seed) is int and 1 <= seed < 2**31 - 1 for seed in SEEDS)
assert MAIN_SEED in SEEDS
print('Seeds:', SEEDS, 'Main seed:', MAIN_SEED)

## Pooled input documents

Pool Normal first and Predatory second, preserving input order and dataset-local IDs. Collapse whitespace. Exclude missing/empty abstracts and abstracts with fewer than ten regex words (`\b\w+\b`). Concatenate title, a period and space, and abstract; use the abstract alone when the title is missing. No publication-year or detector-coverage restriction is applied. One BERTopic model is fit to the pooled corpus per seed.

In [ ]:
papers = []
links = {}
clean_text = lambda value: ' '.join(value.split()) if isinstance(value, str) else ''
for dataset, group in DATASETS.items():
    d = pd.read_csv(DATA_DIR / group / 'papers.csv', usecols=['id','title','description']).rename(columns={'id':'paper_id'})
    assert d.paper_id.notna().all() and d.paper_id.is_unique
    d['dataset'] = dataset
    papers.append(d)
    links[dataset] = pd.read_csv(DATA_DIR / group / 'paper_authors.csv', usecols=['paper_id','author_id']).drop_duplicates()
all_papers = pd.concat(papers, ignore_index=True)
abstracts = all_papers.description.map(clean_text)
titles = all_papers.title.map(clean_text)
eligible = abstracts.ne('') & abstracts.map(lambda value: len(re.findall(r'\b\w+\b', value))).ge(10)
documents = [f'{titles[i]}. {abstracts[i]}' if titles[i] else abstracts[i] for i in all_papers.index[eligible]]
mapping = all_papers.loc[eligible, ['paper_id','dataset']].reset_index(drop=True)
mapping['document_sha256'] = [hashlib.sha256(document.encode('utf-8')).hexdigest() for document in documents]
assert not mapping.duplicated(['dataset','paper_id']).any()
print(mapping.groupby('dataset', sort=False).size().rename('eligible_documents').to_string())

## Optional BERTopic fitting — disabled by default

Per topic, row-normalize the class-TF-IDF representation, use cosine distance and single-linkage clustering, and take each leaf's first-merge distance. Min–max normalize across non-outlier topics; smaller scores indicate broader/overlapping topics. Author distances instead use the model's topic embeddings.

Preserve the source outlier reduction (`distributions`, threshold 0.05). Assigned-topic scores determine broad/specific membership. Continuous scores are probability-weighted sums across the non-outlier topic scores, without renormalizing membership probabilities. Model fitting, document transformation and optional numeric artifact saving occur only when `RUN_TOPIC_MODEL` is explicitly enabled. Full models and topic text are never exported.

In [ ]:
def fit_topic_artifacts(seed):
    if not RUN_TOPIC_MODEL:
        raise RuntimeError('BERTopic fitting is disabled.')
    if sys.version_info[:2] != (3, 13):
        raise RuntimeError('Use the recorded Python 3.13 training environment; the Python 3.11 path is for saved artifacts.')
    import torch
    from bertopic import BERTopic
    from bertopic.vectorizers import ClassTfidfTransformer
    from bertopic.representation import KeyBERTInspired
    from sklearn.feature_extraction.text import CountVectorizer
    from umap import UMAP
    from hdbscan import HDBSCAN
    from scipy.cluster.hierarchy import linkage
    from scipy.spatial.distance import pdist

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    model = BERTopic(
        embedding_model='all-mpnet-base-v2',
        umap_model=UMAP(n_neighbors=15, n_components=10, metric='cosine', random_state=seed),
        hdbscan_model=HDBSCAN(min_cluster_size=50, min_samples=5, metric='euclidean', prediction_data=True),
        vectorizer_model=CountVectorizer(stop_words='english', max_df=.85, min_df=1),
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
        representation_model=KeyBERTInspired(), top_n_words=5,
        nr_topics=30, min_topic_size=50, calculate_probabilities=True, verbose=False)
    initial_topics, _ = model.fit_transform(documents)
    assigned = model.reduce_outliers(documents=documents, topics=initial_topics,
        strategy='distributions', probabilities=model.probabilities_, threshold=.05)
    model_ids = np.asarray(sorted(model.get_topics()), dtype=int)
    non_outlier = model_ids != -1
    counts = model.c_tf_idf_
    totals = np.asarray(counts.sum(axis=1)).ravel()
    totals[totals == 0] = 1
    phi = counts.multiply(1/totals[:,None]).toarray()[non_outlier]
    topic_ids = model_ids[non_outlier]
    tree = linkage(pdist(phi, metric='cosine'), method='single')
    scores = np.zeros(len(topic_ids))
    for first, second, distance, _ in tree:
        for child in (int(first), int(second)):
            if child < len(topic_ids) and scores[child] == 0:
                scores[child] = distance
    span = scores.max() - scores.min()
    if span == 0:
        raise ValueError('Topic distances have no variation.')
    topic_scores = pd.DataFrame({'topic_id':topic_ids, 'hier_score':scores, 'hier_norm':(scores-scores.min())/span})
    _, probabilities = model.transform(documents)
    probability_ids = np.arange(probabilities.shape[1])
    score_map = topic_scores.set_index('topic_id').hier_norm
    assert set(probability_ids).issubset(score_map.index)
    assignments = mapping[['paper_id','dataset']].copy()
    assignments['topic_id'] = assigned
    assignments['weighted_score'] = probabilities.dot(score_map.loc[probability_ids].to_numpy())
    vectors = np.asarray(model.topic_embeddings_)[non_outlier]
    folder = TOPIC_DIR / f'seed_{seed}'
    if folder.exists():
        raise FileExistsError('Refusing to overwrite existing seed artifacts.')
    folder.mkdir(parents=True)
    assignments.to_csv(folder/'assignments.csv', index=False)
    topic_scores.to_csv(folder/'topic_scores.csv', index=False)
    mapping.to_csv(folder/'document_mapping.csv', index=False)
    np.savez(folder/'topic_embeddings.npz', topic_ids=topic_ids, embeddings=vectors)
    return assignments, topic_scores, topic_ids, vectors

def load_topic_artifacts(seed):
    folder = TOPIC_DIR / f'seed_{seed}'
    if not folder.is_dir():
        raise FileNotFoundError('Saved seed artifacts are missing; no automatic fitting is allowed.')
    assignments = pd.read_csv(folder/'assignments.csv', usecols=['paper_id','dataset','topic_id','weighted_score'])
    topic_scores = pd.read_csv(folder/'topic_scores.csv', usecols=['topic_id','hier_score','hier_norm'])
    saved_mapping = pd.read_csv(folder/'document_mapping.csv')
    pd.testing.assert_frame_equal(saved_mapping[mapping.columns], mapping, check_dtype=False)
    pd.testing.assert_frame_equal(assignments[['paper_id','dataset']], mapping[['paper_id','dataset']], check_dtype=False)
    with np.load(folder/'topic_embeddings.npz', allow_pickle=False) as saved:
        topic_ids = saved['topic_ids'].copy()
        vectors = saved['embeddings'].copy()
    return assignments, topic_scores, topic_ids, vectors

## Pooled GMM and paper-level statistics — Section 4.1.2

Following the source training logic, fit a two-component GMM to the pooled **paper-level assigned-topic scores**: each valid paper contributes its topic's normalized score. Thus topics are weighted by their paper counts; this is not one observation per distinct topic. Predatory scores precede Normal scores, matching the source ordering.

Choose the grid point with the smallest posterior-probability difference on 10,001 equally spaced points between the component means. This approximates the equal-posterior cutoff. Scores at or above the cutoff are specific. Saved cutoffs and categories are never used to set the result.

Exclude missing, outlier (-1), and unmapped topic assignments. The continuous-score sample additionally requires a finite saved probability-weighted score; validate that it is the same population. The z-test is one-sided, Normal specific proportion greater than Predatory, as in the source. Cohen's h and paper-level Cohen's d are Normal minus Predatory, with d using the pooled sample standard deviation. No chi-square, alternative cutoff, or additional hypothesis test is computed.

In [ ]:
def cohens_d(first, second):
    first, second = np.asarray(first, dtype=float), np.asarray(second, dtype=float)
    assert min(len(first), len(second)) >= 2
    pooled_sd = np.sqrt(((len(first)-1)*first.var(ddof=1) + (len(second)-1)*second.var(ddof=1))/(len(first)+len(second)-2))
    if pooled_sd <= 0:
        raise ValueError('Pooled standard deviation must be positive.')
    return (first.mean()-second.mean())/pooled_sd

def paper_statistics(seed, assignments, topic_scores):
    assert topic_scores.topic_id.is_unique and not topic_scores.topic_id.eq(-1).any()
    assert np.isfinite(topic_scores[['hier_score','hier_norm']].to_numpy()).all()
    assert topic_scores.hier_norm.between(0, 1).all()
    span = topic_scores.hier_score.max() - topic_scores.hier_score.min()
    assert span > 0
    assert np.allclose(topic_scores.hier_norm, (topic_scores.hier_score-topic_scores.hier_score.min())/span)
    score_map = topic_scores.set_index('topic_id').hier_norm
    assigned_scores = assignments.topic_id.map(score_map)
    valid = assignments.topic_id.notna() & assignments.topic_id.ne(-1) & assigned_scores.notna()
    assert np.isfinite(assignments.loc[valid, 'weighted_score']).all()
    groups = {dataset: assignments.loc[valid & assignments.dataset.eq(dataset)] for dataset in DATASETS}
    hard = {dataset: group.topic_id.map(score_map).to_numpy() for dataset, group in groups.items()}
    pooled = np.concatenate([hard['Predatory'], hard['Normal']]).reshape(-1, 1)
    gmm = GaussianMixture(n_components=2, random_state=seed).fit(pooled)
    grid = np.linspace(gmm.means_.min(), gmm.means_.max(), 10001).reshape(-1, 1)
    posterior = gmm.predict_proba(grid)
    cut = float(grid[np.argmin(np.abs(posterior[:,0]-posterior[:,1])),0])
    n_pred, n_norm = len(hard['Predatory']), len(hard['Normal'])
    k_pred, k_norm = int((hard['Predatory'] >= cut).sum()), int((hard['Normal'] >= cut).sum())
    p_pred, p_norm = k_pred/n_pred, k_norm/n_norm
    pooled_proportion = (k_pred+k_norm)/(n_pred+n_norm)
    z = (p_norm-p_pred)/np.sqrt(pooled_proportion*(1-pooled_proportion)*(1/n_pred+1/n_norm))
    return dict(seed=seed, gmm_cut=cut, pred_n=n_pred, normal_n=n_norm,
        pred_specific_n=k_pred, normal_specific_n=k_norm,
        pred_specific_percent=100*p_pred, normal_specific_percent=100*p_norm,
        paper_h=2*(np.arcsin(np.sqrt(p_norm))-np.arcsin(np.sqrt(p_pred))),
        paper_d=cohens_d(groups['Normal'].weighted_score, groups['Predatory'].weighted_score),
        proportion_z=z, proportion_p_one_sided=norm.sf(z))

## Author consistency — Section 4.2.1

For each dataset separately, join deduplicated paper–author links to valid non-outlier topic assignments. Require at least two distinct valid topics. Compute the unweighted mean of `1 − cosine_similarity` over every distinct topic pair; repeating a topic does not increase its pair weight. The stricter sample additionally requires at least three distinct papers with valid topics, not three distinct topics.

For both inclusion rules, report means, denominators, two-sided Welch's t, two-sided asymptotic Mann–Whitney U, and pooled-standard-deviation Cohen's d. Author t and d are Predatory minus Normal. Author names and per-author results are never printed or exported.

In [ ]:
def author_statistics(seed, assignments, topic_ids, vectors):
    assert len(topic_ids) == len(set(topic_ids)) == len(vectors)
    assert -1 not in topic_ids and np.isfinite(vectors).all()
    assert (np.linalg.norm(vectors, axis=1) > 0).all()
    similarity = cosine_similarity(vectors)
    topic_rows = {int(topic):i for i, topic in enumerate(topic_ids)}
    assert np.allclose(similarity, similarity.T) and np.allclose(np.diag(similarity), 1)
    author_values = {}
    for dataset in DATASETS:
        topics = assignments.loc[assignments.dataset.eq(dataset), ['paper_id','topic_id']]
        joined = links[dataset].merge(topics, on='paper_id', how='inner', validate='many_to_one')
        valid = joined[joined.topic_id.isin(topic_rows) & joined.topic_id.ne(-1)]
        values = {'primary':[], 'min3':[]}
        for _, block in valid.groupby('author_id', sort=False):
            distinct_topics = sorted(block.topic_id.astype(int).unique())
            if len(distinct_topics) < 2:
                continue
            distance = np.mean([1-similarity[topic_rows[first],topic_rows[second]] for first, second in combinations(distinct_topics,2)])
            values['primary'].append(distance)
            if block.paper_id.nunique() >= 3:
                values['min3'].append(distance)
        author_values[dataset] = values
    rows = []
    for rule in ['primary','min3']:
        pred = np.asarray(author_values['Predatory'][rule])
        normal = np.asarray(author_values['Normal'][rule])
        welch = ttest_ind(pred, normal, equal_var=False)
        rank = mannwhitneyu(pred, normal, alternative='two-sided', method='asymptotic')
        rows.append(dict(seed=seed, rule=rule, pred_n=len(pred), normal_n=len(normal),
            pred_mean=pred.mean(), normal_mean=normal.mean(), welch_t=welch.statistic,
            welch_p=welch.pvalue, mann_whitney_u=rank.statistic, mann_whitney_p=rank.pvalue,
            author_d=cohens_d(pred,normal)))
    return rows

## Five-seed results — Appendix A.2 and Table 3

For each saved seed, recompute its GMM cutoff, paper statistics and both author samples. BERTopic fitting runs only in the explicitly enabled branch. Print and save aggregate numeric results; no manuscript values or comparison targets are embedded. Ranges summarize the five runs on the same corpus, not independent replications or a pooled significance test.

In [ ]:
paper_rows, author_rows = [], []
for seed in SEEDS:
    if RUN_TOPIC_MODEL:
        assignments, topic_scores, topic_ids, vectors = fit_topic_artifacts(seed)
    else:
        assignments, topic_scores, topic_ids, vectors = load_topic_artifacts(seed)
    assert set(topic_scores.topic_id.astype(int)) == set(topic_ids)
    paper_rows.append(paper_statistics(seed, assignments, topic_scores))
    author_rows.extend(author_statistics(seed, assignments, topic_ids, vectors))
paper_results = pd.DataFrame(paper_rows)
author_results = pd.DataFrame(author_rows)
table3 = paper_results[['seed','gmm_cut','pred_specific_percent','normal_specific_percent','paper_h','paper_d']].merge(
    author_results[author_results.rule.eq('primary')][['seed','author_d']].rename(columns={'author_d':'author_d_primary'}), on='seed', validate='one_to_one').merge(
    author_results[author_results.rule.eq('min3')][['seed','author_d']].rename(columns={'author_d':'author_d_min3'}), on='seed', validate='one_to_one')
paper_results.to_csv(OUTPUT_DIR/'paper_results.csv', index=False)
author_results.to_csv(OUTPUT_DIR/'author_results.csv', index=False)
table3.to_csv(OUTPUT_DIR/'table3.csv', index=False)
print('Main seed, paper results:')
print(paper_results[paper_results.seed.eq(MAIN_SEED)].to_string(index=False))
print('Main seed, author results:')
print(author_results[author_results.seed.eq(MAIN_SEED)].to_string(index=False))
print('Table 3:')
print(table3.to_string(index=False))
print('Five-seed ranges:')
print(table3[['paper_h','paper_d','author_d_primary','author_d_min3']].agg(['min','max']).to_string())